# Metrics
Measure counts and durations so service behavior is visible.


In [ ]:
import time

requests = 0
started = time.perf_counter()
requests += 1
duration = time.perf_counter() - started
print({"requests": requests, "duration_seconds": duration})


## Polished version
Depend on a metrics interface and measure work with a reusable context manager.


In [ ]:
from collections import defaultdict
from collections.abc import Iterator
from contextlib import contextmanager
from typing import Protocol

class Metrics(Protocol):
    def increment(self, name: str) -> None: ...
    def observe(self, name: str, value: float) -> None: ...

class MemoryMetrics:
    def __init__(self) -> None:
        self.counters: dict[str, int] = defaultdict(int)
        self.observations: dict[str, list[float]] = defaultdict(list)
    def increment(self, name: str) -> None:
        self.counters[name] += 1
    def observe(self, name: str, value: float) -> None:
        self.observations[name].append(value)

@contextmanager
def measure_request(metrics: Metrics) -> Iterator[None]:
    metrics.increment("requests_total")
    started = time.perf_counter()
    try:
        yield
    finally:
        metrics.observe("request_duration_seconds", time.perf_counter() - started)

metrics = MemoryMetrics()
with measure_request(metrics):
    sum(range(100))
print(metrics.counters, metrics.observations)
